# A strong directory, a weak denominator

**What eighteen years of an industry top-100 ranking can (and cannot) tell a manufacturer planning its dealer program**

Alex Jaremko · SignalPath Consulting Group · Google Data Analytics capstone

This notebook mirrors the R Markdown case study at https://github.com/SignalPathStrategy/directory-not-denominator. The data are attached as the Kaggle dataset *CE Pro 100 Rankings 2009-2026 (names stripped)*.

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 5, warn = -1, readr.show_col_types = FALSE)
suppressPackageStartupMessages({
library(tidyverse)
library(scales)
})
# Every table and chart below regenerates from the CSVs in data/. Nothing reaches back into private files.
# On Kaggle the same files are attached as a dataset; everywhere else they sit in data/ beside this file.
d <- if (dir.exists("/kaggle/input")) dirname(list.files("/kaggle/input", "cepro100_2009_2026_rows.csv", recursive = TRUE, full.names = TRUE)[1]) else "data"
rows  <- read_csv(file.path(d, "cepro100_2009_2026_rows.csv"))
A1    <- read_csv(file.path(d, "A1_list_level_stats_by_year.csv"))
A2    <- read_csv(file.path(d, "A2_same_store_vs_list_level_growth.csv"))
A3    <- read_csv(file.path(d, "A3_persistent_cohort_series.csv"))
A4    <- read_csv(file.path(d, "A4_entry_exit_by_year.csv"))
A5    <- read_csv(file.path(d, "A5_size_bands_by_year.csv"))
A6    <- read_csv(file.path(d, "A6_years_on_list_distribution.csv"))
A7    <- read_csv(file.path(d, "A7_status_of_every_firm_ever_listed.csv"))
B1    <- read_csv(file.path(d, "B1_reconstructed_landscape_by_population.csv"))
B2    <- read_csv(file.path(d, "B2_firms_above_size_thresholds.csv"))
B3    <- read_csv(file.path(d, "B3_size_distribution_listed_vs_unlisted.csv"))
m <- function(x) dollar(x, scale = 1e-6, suffix = "M", accuracy = 0.1)

# 1. Introduction

**The Business Case**

Smart home equipment manufacturers spend millions each year on dealer incentives—co-op marketing, tier discounts, and field support—guided largely by the industry’s published Top 100 ranking. SignalPath Consulting Group was engaged to answer a fundamental question for FY2027 planning: Is a voluntary, self-reported ranking robust enough to drive channel allocation, and if not, what should replace it?

**The Client & Context**

The client is a hypothetical firm, representative of high-end smart home manufacturers producing control systems, AV, networking, and lighting gear. Because products reach homeowners strictly through thousands of independent, owner-operated custom integration firms, allocating channel budgets accurately is central to their go-to-market strategy.

**The Measurement Problem**

The issue isn't publisher error or bad intent; it's structural sampling bias. Treating a voluntary opt-in list as a complete market census is a known analytical trap. Assuming the ranking is compiled honestly, we test its structural integrity using three metrics:

-   **Composition:** Comparing same-store median growth (firms listed in consecutive years) against list-level median changes. Divergence indicates the list is measuring survey participation rather than market growth.

-   **Coverage:** Benchmarking the list's share of total firms and revenue against an independent market census across multiple revenue tiers.

-   **Selection:** Comparing listed firms against former listees and never-listed peers using size distributions and revenue per employee to surface hidden bias.

**Data & Sources**

The study evaluates two firm-level datasets spanning 2009 through 2026:

-   **The Top 100 Dataset:** 18 years of published rankings, cleaned and reconciled for mergers, name changes, and print corrections.

-   **The Market Population:** An independent dataset built from buying-group rosters, national certification directories, company websites, and public business records to capture all qualifying integrators regardless of survey participation.

**Working Hypothesis**

The Top 100 is a strong directory and a weak denominator. It works well for locating prominent leads, but it is unreliable for sizing the total market, tracking real growth, or benchmarking dealer tiers.

**Deliverables**

Designed for sales executives, buying-group leaders, and trade analysts, the project delivers a fully audited R Markdown notebook and data-cleaning log for full verification.

# 2. Sourcing and Preparing the Data

This analysis relies on two primary datasets hosted in the public repository (`SignalPathStrategy/directory-not-denominator` on GitHub and Kaggle), backed by private working files to protect non-public firm estimates and identity mappings.

**Core Data Sources**

-   **The CE Pro 100 Ranking (2009–2026):** 18 annual editions covering fiscal years 2008 through 2025. Recent editions (2020–2026) were parsed programmatically from text PDFs; older magazine scans (2009–2019) required custom page-geometry parsers and manual transcription for damaged scans.

-   **The Independent Comparator Population:** A baseline of 282 unique qualifying integrators built from buying-group rosters and national certification directories (June 2026). Cross-referencing this roster against historical lists categorizes the market into active listees, alumni, and never-listed firms.

-   **Unlisted Revenue Estimates:** Revenue for never-listed control firms was modeled using public Paycheck Protection Program (PPP) payroll data, web headcount, and industry press. Estimates carry explicit confidence ratings (High/Medium/Low) and are published strictly in aggregate.

**Data Architecture & Anonymization**

The flat public extract (`cepro100_2009_2026_rows.csv`) contains 1,807 firm-year rows across 379 unique entities and 17 attributes. To maintain research integrity without calling out individual businesses, firm names are replaced with stable IDs, city data is removed, and five major non-custom integration outliers are flagged for exclusion.

**Source Credibility (ROCCC Framework)**

-   **Reliable:** Within limits. All figures are self-reported and unaudited; the publisher's third-party verification badge was dropped post-2019.

-   **Original:** Yes. Direct first-party extraction from publisher archives, not secondary aggregations.

-   **Comprehensive:** No—and that is the core point of the study. Voluntary participation creates a structural selection gap.

-   **Current:** Yes. The 2026 edition captures FY2025 financial figures.

-   **Cited:** Yes. Every data point maps to a specific issue, page number, and line item.

**Data Integrity & Discrepancy Auditing**

-   **Entity Resolution:** Resolved 397 raw company name variations down to 379 distinct firms across 18 years of mergers, rebrands, and name changes.

-   **Publisher Discrepancies:** Line-item table totals regularly diverge from publisher summary cards. For example, CE Pro’s 2024 editorial summary claimed \$6.33 billion in total market revenue, but table line items sum to \$1.80 billion—a \$4.53 billion skew caused by including a single massive security enterprise.

-   **Schema Drift:** Changing magazine layouts over 18 years were unified; missing historical attributes are treated as null values rather than zeros.

In [ ]:
rows %>%
  group_by(list_year) %>%
  summarise(firms = n(), with_revenue = sum(!is.na(revenue_usd)), with_employees = sum(!is.na(employees)),
            with_rmr = sum(!is.na(pct_rmr)), with_outlook = sum(!is.na(business_outlook))) %>%
  knitr::kable(caption = "Rows and field availability by list year")

# 3. Processing the Data: Cleaning, Identity Resolution & Decisions

Most portfolio write-ups gloss over data cleaning, but skipping it here breaks the entire analysis. Because the core findings hinge on all 1,807 rows, every entry had to be mapped to its true owner to verify that printed numbers reflected actual publisher intent.

**Tooling Choices**

-   **Python:** Built for messy extraction. Eighteen years of shifting PDF layouts aren't meant for a spreadsheet: seven recent editions used page-geometry parsing, eleven older magazine layouts required custom scrapers, and cross-year name matching needed reusable code that could re-run whenever matching rules evolved.

-   **Spreadsheet Master (Two Copies):** Kept as a human-review workbook (one full dataset, one with outliers excluded). Every single decision required manual row-by-row inspection with attached context notes.

-   **R Markdown:** Handles the final analysis. Narrative, code, outputs, and logging sit in one dynamic document that rebuilds straight from the public extract. Anyone cloning the repo can verify the results instantly.

**Key Data Prep Steps**

-   **Strings to Numbers:** Stripped currency and formatting from printed strings (e.g., `"$12,345,678"` or `"1,200"`) into raw integers. Unclean or non-numeric print entries were preserved verbatim in a private notes field and left blank in the extract.

-   **Location Normalization:** Extracted state codes from city/state strings, converting legacy wire-service abbreviations into standard two-letter postal codes.

-   **Recalculating Revenue per Employee:** Calculated directly as revenue divided by headcount. The publisher’s printed ratio (2009–2020) regularly contradicted its own revenue and employee columns, so the recalculated metric is used throughout.

-   **Outlook Parsing:** Split free-text outlook notes (`"Up 10%"`, `"Flat"`, `"Down 5%"`) into a directional flag and a numerical percentage.

-   **Revenue Bins:** Grouped every row into fixed buckets (Under \$1M, \$1–2M, up to \$25M+). Used the exact same bins for unlisted companies in Test 3 to keep comparisons direct.

**Identity Resolution (The Hard Part)**

Companies rebrand, merge, get acquired, or drop off and return years later. Cross-year matching relied on three layers:

1.  **Normalized Name Keys:** Stripped case, punctuation, legal suffixes (`Inc`, `LLC`), and `DBA` titles.

2.  **Manual Alias Table:** Applied \~60 documented aliases for rebrands and printing variants.

3.  **Entity Merges:** Executed 18 identity merges after tracing every company that ever left the list. A returning business under a new name is counted as a return, not a fresh market entrant. This step collapsed 397 apparent firms down to 379 true unique businesses.

What wasn't merged mattered just as much. Five pairs of similarly named firms were kept separate because they operated in completely different states, and six more were kept distinct based on clear operational evidence. Forcing those merges would have fabricated fake business longevity. The public export exposes only the final, sequence-assigned `firm_id`.

**Analytical Rulings**

-   **Five Outliers Flagged:** Five entities on the list aren't custom integrators (two national security/smart-home providers, a regional security firm, a commercial AV integrator, and a retail giant's install branch). The largest reported \~1,000 times the median revenue at its peak. They remain in the raw extract, flagged, but are excluded from all baseline calculations.

-   **2025 Correction:** Incorporated the publisher's post-print correction, inserting four omitted companies into their true revenue spots and re-ranking the bottom entries.

-   **Misprints Preserved:** Geocoding errors were corrected only in derived state fields, leaving original printed text intact. Arithmetic glitches and duplicated rows were flagged rather than overwritten.

-   **Ranking Quirks Maintained:** Skips after ties (e.g., 33, 33, 35), duplicated ranks, and out-of-order revenue entries were kept exactly as published. Ranks reflect publisher output, not custom re-sorting.

**Validation & Audit**

-   Revenue totals reconcile to the printed tables for all 18 editions (not to the publisher's summary cards, which disagree with its own tables — see Section 2).

-   Zero duplicate `firm_id` entries exist within any single year.

-   Automated text searches verified that no real company names slipped into the public exports.

-   The full decision log lives in `02_cleaning_and_decision_log.md` within the repository.

In [ ]:
# The five outliers are the only firms named anywhere in this package (see the cleaning log for why).
outlier_names <- tribble(
  ~firm_id, ~company,                    ~what_it_is,
  "F0131",  "ADT",                       "National security and smart-home service provider",
  "F0135",  "Vivint",                    "National security and smart-home service provider",
  "F0001",  "Guardian Protection",       "Regional security company",
  "F0269",  "CCS Presentation Systems",  "Commercial-AV integrator",
  "F0153",  "Best Buy",                  "Consumer-electronics retailer's custom-installation line"
)
rows %>%
  filter(outlier_excluded_from_excl_metrics == "Y") %>%
  group_by(firm_id) %>%
  summarise(years_on_list = n(), peak_revenue = max(revenue_usd, na.rm = TRUE)) %>%
  left_join(outlier_names, by = "firm_id") %>%
  arrange(desc(peak_revenue)) %>%
  mutate(peak_revenue = m(peak_revenue)) %>%
  select(firm_id, company, what_it_is, years_on_list, peak_revenue) %>%
  knitr::kable(caption = "The five excluded outliers — kept in the extract, removed from every 'excl.' metric")

# 4. Data Analysis: Three Key Tests

**Dataset Setup**

The 1,807 rows were reshaped for three distinct tests:

-   **List-level stats:** Grouped by publication year.

-   **Same-store growth:** Paired year-over-year rows (`Year N` matched to `Year N-1` by `firm_id`), keeping only firms present in both consecutive years.

-   **Coverage & Selection:** Tagged across five populations (Printed List, Active Alumni w/ Estimates, Active Alumni Carried Forward, Buying-Group Unlisted, Certified Unlisted).

All metrics exclude the five giant outliers unless noted. *Same-store growth* functions like retail store metrics: tracking performance exclusively among firms present in both comparison periods so new entries and exits don't skew growth signals.

## Test 1 — Composition: Does Year-Over-Year List Change Equal Market Growth?

Headline list shifts track *who showed up*, not how companies actually grew.

In 2026, among the 56 firms listed in both 2025 and 2026, **same-store median revenue grew +8.0%**. Meanwhile, the **list's overall median revenue dropped -31.9%** (falling from \$6.6M to \$4.5M). Both figures are accurate, but the overall median drop isn't a market contraction—it reflects a shift in participation. The 2026 list saw 43 entrants (37 first-time) and 42 exits. Retention plummeted to 58% (down from 79%–88% over the prior five years). The incoming cohort was simply smaller, pulling down the average list benchmark.

This churn isn't new:

-   **2010:** Same-store growth was -16.0% while list medians dropped -30.5%.

-   **2020:** Same-store grew +5.8% while list totals surged +33.6% due to new enterprise joins.

Tracking the 6 non-outlier firms present on all 18 lists shows a steady compound annual growth rate (CAGR) of \~6.9% in aggregate revenue (\$72.6M in 2009 to \$225.6M in 2026), with median revenue growing \~5.4% annually (\$7.0M to \$16.9M). Across the full dataset, 119 of 379 firms appeared only once.

In [ ]:
A2 %>%
  select(list_year, `Same-store (matched firms)` = same_store_median_growth, `List-level (median to median)` = list_level_median_change) %>%
  pivot_longer(-list_year, names_to = "series", values_to = "growth") %>%
  ggplot(aes(list_year, growth, colour = series)) +
  geom_hline(yintercept = 0, colour = "grey60") +
  geom_line(linewidth = 1) + geom_point() +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  scale_x_continuous(breaks = 2010:2026) +
  labs(title = "Two ways to read 'growth' from the same list", subtitle = "Median revenue change, excluding the five outliers",
       x = "List year (fiscal year is one earlier)", y = NULL, colour = NULL) +
  theme_minimal() + theme(legend.position = "top")

In [ ]:
A2 %>%
  transmute(list_year, matched_firms, `same-store median` = percent(same_store_median_growth, 0.1),
            `same-store total` = percent(same_store_total_growth, 0.1), `list-level median` = percent(list_level_median_change, 0.1),
            `list-level total` = percent(list_level_total_change, 0.1)) %>%
  knitr::kable(caption = "Same-store versus list-level growth, by list year")

In [ ]:
A4 %>% filter(!is.na(new_entrants)) %>%
  select(list_year, `New entrants` = new_entrants, Exits = exits) %>%
  pivot_longer(-list_year) %>%
  ggplot(aes(list_year, value, fill = name)) + geom_col(position = "dodge") +
  scale_x_continuous(breaks = 2010:2026) +
  labs(title = "Who showed up: entries and exits each year", x = NULL, y = "Firms", fill = NULL) +
  theme_minimal() + theme(legend.position = "top")

In [ ]:
A3 %>% mutate(across(ends_with("usd"), m)) %>%
  knitr::kable(caption = "The persistent cohort — firms present on all eighteen lists")

## Test 2 — Coverage: How Much Market Volume Does the List Capture?

The list captures only a small fraction of the viable dealer universe—about **24% of core firms (98 of 411)** and **27% of core revenue (\$785M out of \$2.93B)**.

Across revenue thresholds, capture rates remain flat around \~25%:

-   **\$1M+ firms:** 25% captured

-   **\$5M+ firms:** 26% captured (43 of 163)

-   **\$10M+ firms:** 28% captured (19 of 68)

-   **\$25M+ firms:** 19% captured (3 of 16)

The list doesn't concentrate on top-tier players while ignoring small operators; it captures roughly one-quarter of the market across every size tier. If every core firm entered, the true #100 company would sit at \~\$7.5M revenue, rather than the list’s current \$1.04M floor, leaving only 26 current listees in the actual Top 100.

In [ ]:
B1 %>% mutate(across(ends_with("usd"), m)) %>%
  knitr::kable(caption = "The reconstructed landscape by population (point estimates with low/high range)")

In [ ]:
B1 %>% filter(!population %in% c("ALL (pro forma ranking)", "CORE (excluding speculative carry-forwards)")) %>%
  mutate(population = fct_reorder(population, total_point_usd)) %>%
  ggplot(aes(population, total_point_usd)) +
  geom_col(fill = "steelblue") +
  geom_errorbar(aes(ymin = total_low_usd, ymax = total_high_usd), width = 0.25) +
  coord_flip() + scale_y_continuous(labels = m) +
  labs(title = "Where the revenue actually sits", subtitle = "The printed list is one of five populations",
       x = NULL, y = "Fiscal-2025 revenue (point estimate; bar = low/high range)") +
  theme_minimal()

In [ ]:
B2 %>% mutate(threshold_usd = m(threshold_usd), share_of_core_firms_captured_by_list = percent(share_of_core_firms_captured_by_list, 1)) %>%
  knitr::kable(caption = "At every size threshold, the list holds roughly a quarter of the firms")

## Test 3 — Selection: Are Listed Firms Built Differently?

Dropping off the list rarely signals business decline. Median revenue breaks down as:

-   **Printed List:** \$4.5M

-   **Active Alumni:** \$7.3M

-   **Never-Listed Roster Firms:** \$2.5M

Active alumni are systematically *larger* than currently listed firms: 63 active alumni generate \$10M+ (with 18 above \$25M), compared to just 19 and 3 on the printed list. Fully 77% of active alumni sit above \$5M, compared to 44% of current listees.

Where the list shines is internal productivity metrics. **Median revenue per employee** among listed firms rose from \$187.5k (2009) to \$275.2k (2026). Because this is a within-firm ratio rather than an aggregate sum, it remains highly stable regardless of member churn.

In [ ]:
B3 %>%
  ggplot(aes(revenue_band, share, fill = population)) +
  geom_col(position = "dodge") +
  scale_y_continuous(labels = percent_format(1)) +
  labs(title = "Size distribution: listed versus unlisted firms", x = NULL, y = "Share of population", fill = NULL) +
  theme_minimal() + theme(legend.position = "top", axis.text.x = element_text(angle = 20, hjust = 1))

In [ ]:
A1 %>% filter(population == "excl_five_outliers") %>%
  ggplot(aes(list_year, median_rev_per_employee_usd)) + geom_line(linewidth = 1) + geom_point() +
  scale_y_continuous(labels = dollar_format(scale = 1e-3, suffix = "K")) + scale_x_continuous(breaks = 2009:2026) +
  labs(title = "Median revenue per employee among listed firms", x = NULL, y = NULL) + theme_minimal()

# 5. Strategic Synthesis: Directory vs. Denominator

The dataset proves one core thesis: **The list functions well as a trade directory, but fails as an analytical denominator.**

-   **As a Directory:** Excellent reference asset. 379 firms over 18 years with documented locations, team sizes, and reported figures.

-   **As a Market Denominator:** Fails due to voluntary participation. It captures \~27% of sector revenue (Coverage Bias), mixes churn with actual business performance (Composition Bias), and samples only firms willing to submit data (Selection Bias).

Looking at all 374 non-outlier firms ever listed: 98 are currently listed, **201 are active non-participants**, 34 were acquired, 21 closed, 12 merged/rebranded, and 8 are untraceable. Having more than double the current list operating as active non-participants underscores why the list cannot serve as a stand-alone market benchmark.

# 6. Practical Solutions & Trade-Offs

**The Business Risks of Using the Unadjusted List**

1.  **Undersized Channel Sizing:** Basing dealer program budgets or rep territories on list totals underestimates market potential by 3x to 4x.

2.  **Misreading Market Trends:** Misinterpreting list churn as economic contractions (e.g., treating 2025's -32% participation drop as a market crash when active firms grew +8%).

3.  **Skewed Dealer Benchmarks:** Rewarding small participants ("Top 100" at \$1.5M) while missing \$12M unlisted dealers entirely.

**Three Strategic Options**

-   **Option A: Build an Independent Dealer Universe (Recommended)**

    -   *Approach:* Combine certification rosters, buying group lists, and alumni data. Estimate missing revenues using documented ratios and confidence ratings.

    -   *Pros:* \~4x coverage improvement; transparent confidence bands; repeatable annual workflow.

    -   *Cons:* Requires initial analyst workload (2–3 weeks upfront).

-   **Option B: Use Third-Party Industry Association Data for Sizing**

    -   *Approach:* Adopt trade association macro numbers for sizing while using the printed list purely for target outreach.

    -   *Pros:* Zero internal data builds; instant executive buy-in.

    -   *Cons:* Macro figures cover \~20,000 broad market entities (median revenue \<\$1M), misaligned with the high-value dealer channel.

-   **Option C: Restrict List Use to Same-Store Growth Only**

    -   *Approach:* Immediate policy rule prohibiting headline list totals; evaluate growth exclusively via matched cohorts.

    -   *Pros:* Zero cost; fixes growth-rate misinterpretation overnight.

    -   *Cons:* Leaves coverage and tiering biases completely unaddressed.

# 7. Recommendations & Immediate Actions

-   **Adopt Option A for FY27 Planning:** Commission an independent dealer universe before Q4 budget locks.

-   **Implement Option C Immediately:** Mandate that all immediate internal reports evaluate list performance on a same-store basis only.

-   **Target High-Value Alumni:** Treat the 201 active former listees as an explicit sales conquest segment.

**Further exploration**

Additional data and deliverables that would extend the findings:

-   **A second research pass on the Low-confidence estimates,** to narrow the coverage range. The conclusion holds at the low end already; a tighter range makes it more useful for budgeting.

-   **A third independent roster** — another buying group or a manufacturer's own authorized-dealer list — to test whether the roughly one-quarter coverage ratio holds against a differently selected comparator.

-   **A licensable roster-match table:** the list-to-roster match, refreshed annually, as a product for manufacturers and buying groups.

-   **A brand-share analysis from the lists' brand pages,** which the extract omits. The editions report the brands each firm carries; matched to the firm-year table, that is a channel-share series by product category.

-   **An annual refresh of the extract each list season,** so the same-store and coverage series extend by one year with a day's work rather than a rebuild.

# **Technical Appendix & Reproducibility Notes**

-   **Regeneration:** Open `case_study.Rmd` in RStudio or run `rmarkdown::render("case_study.Rmd")`. Requires R 4.x (`tidyverse`, `knitr`, `rmarkdown`).

-   **Public vs. Private:** Public repository includes cleaned anonymized extracts (`firm_id`), aggregated tables (A1–A7, B1–B3), and decision logs. Raw identity mapping tables, unmasked company names, and proprietary firm-level estimates remain private.

-   **AI Usage Disclosure:** Generative tools assisted in writing scraper logic, running preliminary desk research verification, and polishing prose. All mapping logic, data rulings, analytical test designs, and strategic conclusions were authored and verified manually by the analyst.

In [ ]:
sessionInfo()